# Notebook 03 — Hotspot Analysis (Getis-Ord Gi*)
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Identify statistically significant clusters of high water stress (hotspots) and low stress (coldspots) across Kitui wards using the Getis-Ord Gi* statistic.

**Requires:** `kitui_wasi_ward_table.csv` from Notebook 02.

---
## Method
- **Spatial weights:** Queen contiguity (wards sharing an edge or vertex)
- **Statistic:** Getis-Ord Gi* (local spatial autocorrelation)
- **Significance threshold:** p < 0.05 (z-score ±1.96)
- **Cluster types:**
  - HH (High-High): High-stress ward surrounded by high-stress neighbours — **priority hotspot**
  - LL (Low-Low): Low-stress ward surrounded by low-stress neighbours — coldspot
  - Not significant: No meaningful spatial clustering

---
## Outputs
- `kitui_hotspot_ward.geojson` — ward polygons with Gi* z-score, p-value, cluster type
- `kitui_hotspot_map.png` — print-ready cluster map
- `kitui_hotspot_summary.csv` — ward-level hotspot table for the report

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install geopandas esda libpysal splot matplotlib folium -q

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

import libpysal
from libpysal.weights import Queen
import esda
from esda.getisord import G_Local

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Kitui_WASHLAB/'
OUT   = DRIVE + 'outputs/'
WGS84 = 'EPSG:4326'

print('Setup complete — run after Notebook 02')

In [ ]:
# ── 1. Load WASI ward results ─────────────────────────────────────────────────

wards = gpd.read_file(OUT + 'kitui_wasi_ward.geojson')
wasi_table = pd.read_csv(OUT + 'kitui_wasi_ward_table.csv')

# Join full component data back to geodataframe
wards = wards.merge(wasi_table.drop(columns=['WASI_mean', 'Stress_Class'], errors='ignore'),
                    on='Ward', how='left')

# Drop any duplicate geometry column
if wards.crs is None:
    wards = wards.set_crs(WGS84)

print(f'Loaded {len(wards)} wards')
print(f'WASI range: {wards["WASI_mean"].min():.3f} – {wards["WASI_mean"].max():.3f}')
print(f'Missing WASI: {wards["WASI_mean"].isna().sum()}')

In [ ]:
# ── 2. Build spatial weights matrix ───────────────────────────────────────────

# Project to UTM for geometrically accurate contiguity detection
UTM_CRS = 'EPSG:32637'
wards_utm = wards.to_crs(UTM_CRS).reset_index(drop=True)

# Queen contiguity — wards sharing at least one point (edge or vertex)
w = Queen.from_dataframe(wards_utm, use_index=False)
w.transform = 'r'  # row-standardise for Gi*

print(f'Spatial weights built: {w.n} units')
print(f'Average neighbours per ward: {w.mean_neighbors:.1f}')
print(f'Islands (wards with no neighbours): {len(w.islands)}')
if w.islands:
    print('  WARNING: The following wards are isolated — check boundary file:')
    for idx in w.islands:
        print(f'    {wards_utm.loc[idx, "Ward"]}')

In [ ]:
# ── 3. Compute Gi* statistic ──────────────────────────────────────────────────

# Fill any NaN WASI values with county mean before running Gi*
wasi_values = wards_utm['WASI_mean'].fillna(wards_utm['WASI_mean'].mean()).values

# Getis-Ord Gi* (star = includes the focal unit in the calculation)
gi_star = G_Local(wasi_values, w, star=True, permutations=999)

wards_utm['Gi_Zs']    = gi_star.Zs      # z-score
wards_utm['Gi_Pval']  = gi_star.p_sim   # simulated p-value
wards_utm['Gi_EV']    = gi_star.EG_sim  # expected value

print('Gi* computed')
print(f'Z-score range: {gi_star.Zs.min():.2f} to {gi_star.Zs.max():.2f}')
print(f'Significant (p<0.05): {(gi_star.p_sim < 0.05).sum()} of {len(wards_utm)} wards')

In [ ]:
# ── 4. Classify hotspot / coldspot clusters ───────────────────────────────────

def classify_hotspot(z, p, threshold=0.05):
    """
    Classify Gi* result into hotspot/coldspot categories.
    Uses three significance levels matching standard GIS practice.
    """
    if p > threshold:
        return 'Not significant'
    if z >= 2.576:   return 'Hotspot (99%)'     # p < 0.01
    if z >= 1.960:   return 'Hotspot (95%)'     # p < 0.05
    if z >= 1.645:   return 'Hotspot (90%)'     # p < 0.10 (informational)
    if z <= -2.576:  return 'Coldspot (99%)'
    if z <= -1.960:  return 'Coldspot (95%)'
    if z <= -1.645:  return 'Coldspot (90%)'
    return 'Not significant'

wards_utm['Hotspot_Class'] = [
    classify_hotspot(z, p)
    for z, p in zip(wards_utm['Gi_Zs'], wards_utm['Gi_Pval'])
]

# Simplified 3-class for mapping
def hotspot_simple(cls):
    if 'Hotspot' in cls:  return 'Hotspot (high stress cluster)'
    if 'Coldspot' in cls: return 'Coldspot (low stress cluster)'
    return 'Not significant'

wards_utm['Hotspot_Simple'] = wards_utm['Hotspot_Class'].apply(hotspot_simple)

print('Hotspot classification:')
print(wards_utm['Hotspot_Class'].value_counts().to_string())
print()
print('Priority hotspot wards (HH at 95%+):')
hotspots = wards_utm[wards_utm['Hotspot_Class'].isin(['Hotspot (99%)', 'Hotspot (95%)'])]
print(hotspots[['Ward','WASI_mean','Gi_Zs','Hotspot_Class']]
      .sort_values('Gi_Zs', ascending=False)
      .to_string(index=False))

In [ ]:
# ── 5. Overlay: hotspots vs borehole infrastructure ───────────────────────────
#
# Key question: which hotspot wards have very few functional boreholes?
# These are the most urgent intervention targets.

wards_utm['Infrastructure_Gap'] = (
    (wards_utm['Hotspot_Simple'] == 'Hotspot (high stress cluster)') &
    (wards_utm['Functional_BH_Count'] <= 3)
)

priority_wards = wards_utm[wards_utm['Infrastructure_Gap'] == True]
print(f'HIGH-PRIORITY wards — hotspot AND ≤3 functional boreholes: {len(priority_wards)}')
if len(priority_wards) > 0:
    print(priority_wards[['Ward', 'WASI_mean', 'Gi_Zs', 'Functional_BH_Count']]
          .sort_values('WASI_mean', ascending=False)
          .to_string(index=False))

In [ ]:
# ── 6. Export hotspot results ─────────────────────────────────────────────────

# Reproject back to WGS84 for GeoJSON export
wards_wgs = wards_utm.to_crs(WGS84)

# GeoJSON for Streamlit app
geojson_cols = ['Ward', 'WASI_mean', 'Gi_Zs', 'Gi_Pval',
                'Hotspot_Class', 'Hotspot_Simple', 'Infrastructure_Gap',
                'Functional_BH_Count', 'geometry']
wards_wgs[geojson_cols].to_file(OUT + 'kitui_hotspot_ward.geojson', driver='GeoJSON')
print(f'GeoJSON saved: {OUT}kitui_hotspot_ward.geojson')

# CSV table for report
csv_cols = ['Ward', 'WASI_mean', 'Stress_Class', 'Gi_Zs', 'Gi_Pval',
            'Hotspot_Class', 'Functional_BH_Count', 'Infrastructure_Gap']
# Re-add Stress_Class from earlier
wards_wgs['Stress_Class'] = wards_wgs['WASI_mean'].apply(
    lambda s: 'Very High' if s >= 0.70 else
              'High' if s >= 0.55 else
              'Moderate' if s >= 0.40 else
              'Low' if s >= 0.25 else 'Very Low'
)
wards_wgs[csv_cols].sort_values('Gi_Zs', ascending=False).to_csv(
    OUT + 'kitui_hotspot_summary.csv', index=False
)
print(f'CSV saved: {OUT}kitui_hotspot_summary.csv')

In [ ]:
# ── 7. Hotspot map ────────────────────────────────────────────────────────────

HOTSPOT_COLOURS = {
    'Hotspot (99%)':  '#C00000',
    'Hotspot (95%)':  '#FF4444',
    'Hotspot (90%)':  '#FF9999',
    'Not significant':'#D9D9D9',
    'Coldspot (90%)': '#9DC3E6',
    'Coldspot (95%)': '#2E75B6',
    'Coldspot (99%)': '#0B5394',
}

fig, axes = plt.subplots(1, 2, figsize=(20, 14))

# Panel A: Gi* z-score continuous
ax = axes[0]
wards_wgs.plot(
    column='Gi_Zs',
    cmap='RdBu_r',
    linewidth=0.5,
    edgecolor='white',
    legend=True,
    legend_kwds={'label': 'Gi* Z-score', 'orientation': 'vertical'},
    ax=ax
)
ax.set_title('Gi* Z-score (continuous)\nPositive = water stress cluster', fontsize=11)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

# Panel B: Classified hotspot map
ax = axes[1]
for cls, colour in HOTSPOT_COLOURS.items():
    subset = wards_wgs[wards_wgs['Hotspot_Class'] == cls]
    if len(subset) > 0:
        subset.plot(ax=ax, color=colour, linewidth=0.5, edgecolor='white')

# Label hotspot wards
for _, row in wards_wgs[wards_wgs['Hotspot_Class'].str.startswith('Hotspot')].iterrows():
    c = row.geometry.centroid
    ax.annotate(row['Ward'], xy=(c.x, c.y), fontsize=6, ha='center', va='center',
                fontweight='bold', color='white')

# Legend
legend_patches = [
    mpatches.Patch(color=HOTSPOT_COLOURS['Hotspot (99%)'],  label='Hotspot (p<0.01)'),
    mpatches.Patch(color=HOTSPOT_COLOURS['Hotspot (95%)'],  label='Hotspot (p<0.05)'),
    mpatches.Patch(color=HOTSPOT_COLOURS['Not significant'],label='Not significant'),
    mpatches.Patch(color=HOTSPOT_COLOURS['Coldspot (95%)'], label='Coldspot (p<0.05)'),
    mpatches.Patch(color=HOTSPOT_COLOURS['Coldspot (99%)'], label='Coldspot (p<0.01)'),
]
ax.legend(handles=legend_patches, loc='lower left', fontsize=9, title='Cluster Type')
ax.set_title('Water Stress Hotspot Classification\nGetis-Ord Gi* | Queen contiguity | 999 permutations',
             fontsize=11)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

plt.suptitle('Spatial Water Stress Clusters — Kitui County, Kenya',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT + 'kitui_hotspot_map.png', dpi=200, bbox_inches='tight')
plt.show()
print('Hotspot map saved')

In [ ]:
# ── 8. Interactive folium map ─────────────────────────────────────────────────

import folium

FOLIUM_COLOURS = {
    'Hotspot (high stress cluster)': '#C00000',
    'Not significant':               '#D9D9D9',
    'Coldspot (low stress cluster)': '#2E75B6',
}

m = folium.Map(location=[-1.3, 38.0], zoom_start=8, tiles='CartoDB positron')

def style_fn(feature):
    cls = feature['properties']['Hotspot_Simple']
    return {
        'fillColor':   FOLIUM_COLOURS.get(cls, '#D9D9D9'),
        'color':       '#555555',
        'weight':      0.8,
        'fillOpacity': 0.65,
    }

def popup_fn(feature):
    p = feature['properties']
    return folium.Popup(
        f"<b>{p['Ward']}</b><br>"
        f"WASI: {p['WASI_mean']:.3f}<br>"
        f"Gi* Z: {p['Gi_Zs']:.2f} | p={p['Gi_Pval']:.3f}<br>"
        f"Cluster: {p['Hotspot_Class']}<br>"
        f"Functional BH: {p['Functional_BH_Count']}",
        max_width=280
    )

gjson = folium.GeoJson(
    wards_wgs[geojson_cols].__geo_interface__,
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(['Ward', 'Hotspot_Class', 'WASI_mean']),
    popup=folium.GeoJsonPopup(['Ward', 'WASI_mean', 'Gi_Zs', 'Hotspot_Class'])
)
gjson.add_to(m)

folium.LayerControl().add_to(m)
m.save(OUT + 'kitui_hotspot_interactive.html')
m

In [ ]:
# ── 9. Summary statistics for report ─────────────────────────────────────────

total = len(wards_wgs)
n_hotspot = len(wards_wgs[wards_wgs['Hotspot_Simple'] == 'Hotspot (high stress cluster)'])
n_coldspot = len(wards_wgs[wards_wgs['Hotspot_Simple'] == 'Coldspot (low stress cluster)'])
n_ns       = len(wards_wgs[wards_wgs['Hotspot_Simple'] == 'Not significant'])
n_infra_gap = wards_wgs['Infrastructure_Gap'].sum()

print('══ HOTSPOT ANALYSIS SUMMARY ══════════════════════════════════════')
print(f'Total wards analysed:          {total}')
print(f'Water stress hotspots (p<0.05):{n_hotspot} ({n_hotspot/total*100:.0f}%)')
print(f'Coldspots (p<0.05):            {n_coldspot} ({n_coldspot/total*100:.0f}%)')
print(f'Not significant:               {n_ns} ({n_ns/total*100:.0f}%)')
print(f'Priority (hotspot + ≤3 BH):    {n_infra_gap}')
print('==================================================================')
print()
print('Moran\'s I spatial autocorrelation (global):')

from esda.moran import Moran
mi = Moran(wasi_values, w)
print(f'  I = {mi.I:.4f} | p = {mi.p_sim:.4f} | z = {mi.z_norm:.3f}')
if mi.p_sim < 0.05:
    direction = 'positive (clustering)' if mi.I > 0 else 'negative (dispersion)'
    print(f'  Significant {direction} spatial autocorrelation in water stress')
else:
    print('  No significant global spatial autocorrelation detected')

print()
print('── Notebook 03 complete ──────────────────────────────────────────')
print('Next: Run Notebook 04 (Coverage Gap Analysis)')